# Preprocesamiento de Datos

En este notebook consideraremos detalles de escalamiento, log-transform, codificacion, etc. 


In [1]:
# imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, FunctionTransformer, PowerTransformer

### Cargamos el dataset

In [2]:
df = pd.read_csv("../data/processed/customer_features.csv", index_col=0)
df.head()

,Pais Principal,Paises Distintos,Permanencia,Compras,Canasta_Prom,Ticket_Prom,Precio_Prom,Precio_Max,Productos Distintos,Pct_Devoluciones
CustomerID,,,,,,,,,,
12347.0,Iceland,1,403.0,8,370.875,615.19125,1.658756,12.75,126,0.00
12348.0,Finland,1,364.0,5,542.800,403.88000,0.744068,40.00,25,0.00
12349.0,Italy,1,572.0,4,406.000,1107.17250,2.727026,300.00,138,0.25
12350.0,Norway,1,1.0,1,197.000,334.40000,1.697462,40.00,17,0.00
12351.0,Unspecified,1,1.0,1,261.000,300.93000,1.152989,12.75,21,0.00


## Transformación log-normal / potencia

Aplicamos transformacion log normal dado que los histogramas no parecen normales y afectaria los resultados del modelos de densidad

In [3]:
df_transformed = df.copy()
transformers = dict()

cols_log_normal = [
    "Compras",
    "Canasta_Prom", "Ticket_Prom", "Precio_Prom", "Precio_Max",
    "Productos Distintos"
]
for col in cols_log_normal:
    transformer = FunctionTransformer(func=np.log1p, inverse_func=np.expm1, validate=True)
    transformer.fit(df_transformed[[col]].to_numpy())

    transformers[col] = transformer

cols_pw_normal = [
    "Pct_Devoluciones"
]
for col in cols_pw_normal:
    transformer = PowerTransformer(method='yeo-johnson', standardize=False)
    transformer.fit(df_transformed[[col]].to_numpy())

    transformers[col] = transformer


for col in [*cols_log_normal, *cols_pw_normal]:
    df_transformed[col] = transformers[col].transform(df_transformed[[col]].to_numpy())

df_transformed.describe()

,Paises Distintos,Permanencia,Compras,Canasta_Prom,Ticket_Prom,Precio_Prom,Precio_Max,Productos Distintos,Pct_Devoluciones
count,5876.000000,5876.000000,5876.000000,5876.000000,5876.000000,5876.000000,5876.000000,5876.000000,5876.000000
mean,1.002212,274.287270,1.549355,4.999872,5.621071,1.090667,2.686656,3.760129,0.065266
std,0.046988,258.997643,0.809555,0.921822,0.732573,0.465508,0.883648,1.219075,0.081191
min,1.000000,1.000000,0.693147,0.693147,0.000000,0.000000,0.000000,0.693147,0.000000
25%,1.000000,1.000000,0.693147,4.530256,5.189364,0.887146,2.251292,2.995732,0.000000
50%,1.000000,221.500000,1.386294,5.049856,5.641640,1.040241,2.621039,3.828641,0.000000
75%,1.000000,513.000000,2.079442,5.556828,6.032868,1.208729,2.887590,4.644391,0.147073
max,2.000000,739.000000,5.988961,11.375593,9.605470,9.301506,9.301506,7.844241,0.217507


## Codificación de variables categóricas

En vez de un one-hot completo del pais (con mas de 40 categorias), resumimos "Pais Principal" y "Paises Distintos" en dos banderas binarias mas informativas para el modelo

In [4]:
df_transformed["Comprador Local"] = (df_transformed["Pais Principal"] == 'United Kingdom').astype(int)
df_transformed["Nomada"] = (df_transformed["Paises Distintos"] > 1).astype(int)

df_transformed.head()

,Pais Principal,Paises Distintos,Permanencia,Compras,Canasta_Prom,Ticket_Prom,Precio_Prom,Precio_Max,Productos Distintos,Pct_Devoluciones,Comprador Local,Nomada
CustomerID,,,,,,,,,,,,
12347.0,Iceland,1,403.0,2.197225,5.918558,6.423557,0.977858,2.621039,4.844187,0.000000,0,0
12348.0,Finland,1,364.0,1.791759,6.298582,6.003591,0.556220,3.713572,3.258097,0.000000,0,0
12349.0,Italy,1,572.0,1.609438,6.008813,7.010468,1.315611,5.707110,4.934474,0.139574,0,0
12350.0,Norway,1,1.0,0.693147,5.288267,5.815324,0.992311,3.713572,2.890372,0.000000,0,0
12351.0,Unspecified,1,1.0,0.693147,5.568345,5.710195,0.766857,2.621039,3.091042,0.000000,0,0


## Escalamiento

Escalamos unicamente las variables numericas continuas; las banderas binarias se dejan sin escalar

In [5]:
# Definimos las columnas numéricas continuas
columnas_numericas = ['Permanencia', 'Compras', 'Canasta_Prom', 'Ticket_Prom',
                       'Precio_Prom', 'Precio_Max', 'Productos Distintos',
                       'Pct_Devoluciones']

# Escalamos solo las numéricas continuas
scaler = StandardScaler()
df_num_escalado = pd.DataFrame(
    scaler.fit_transform(df_transformed[columnas_numericas]),
    columns=columnas_numericas,
    index=df_transformed.index
)

df_cat_escalado = df_transformed[["Comprador Local", "Nomada"]]

# Dataset final para los modelos
df_final = pd.concat([df_num_escalado, df_cat_escalado], axis=1)

print(df_final.shape)
df_final.head()

(5876, 10)


,Permanencia,Compras,Canasta_Prom,Ticket_Prom,Precio_Prom,Precio_Max,Productos Distintos,Pct_Devoluciones,Comprador Local,Nomada
CustomerID,,,,,,,,,,
12347.0,0.497007,0.800347,0.996682,1.095527,-0.242354,-0.074263,0.889322,-0.803925,0,0
12348.0,0.346414,0.299455,1.408970,0.522203,-1.148191,1.162232,-0.411849,-0.803925,0,0
12349.0,1.149578,0.074223,1.094601,1.896758,0.483263,3.418458,0.963390,0.915303,0,0
12350.0,-1.055263,-1.057718,0.312880,0.265187,-0.211304,1.162232,-0.713517,-0.803925,0,0
12351.0,-1.055263,-1.057718,0.616736,0.121669,-0.695665,-0.074263,-0.548894,-0.803925,0,0


## Guardar dataset preprocesado

In [6]:
df_final.to_csv("../data/processed/customer_features_model.csv", encoding="utf-8")